## Performs the initial summary and checking of the Attention Shift

This script does cross-checking for consistency after the initial `_events_temp1.tsv`
files are produced. The `as_01_initial_combination.ipnb` script performs
the initial file production.  If there are errors or inconsistencies in this
result, then additional data cleaning will need to be performed on the input
files and everything rerun.

## The following should be removed

`sub-020_run-01`, `sub-021_run-01`, and `sub-022_run-01` should be deleted and the
corresponding run-02 files renamed as run-01.  The original run-01 files only contain
pulse codes.

The `sub-007_run-01` has 3 events with `event_code` value of 255.  These should
be removed.

### Checking for forbidden codes
       Codes 1 and 2 can appear anywhere
       Codes 3 through 6 should appear only in the focus condition.
       Codes 7 through 14 should appear only in the shift condition.
       Codes 199, 201, 202, and 255 are not related to condition.

In [1]:
from hed.tools.hed_logger import HedLogger
from hed.tools.io_utils import get_file_list, make_file_dict
from hed.tools.data_utils import get_new_dataframe
from hed.tools.map_utils import make_combined_dicts

# Set up the logger
status = HedLogger()

# Make the dictionaries of the events.tsv files and the EEG.set events files
bids_root_path = 'G:\AttentionShift\AttentionShiftWorking'
bids_files = get_file_list(bids_root_path, extensions=[".tsv"], name_suffix="_events_temp1")
bids_dict = make_file_dict(bids_files, indices=(0, 2))
bids_skip_cols = ['onset', 'duration', 'sample', 'latency', 'sample_offset']

print('\nBIDS events summary:')
bids_dicts_all, bids_dicts =  make_combined_dicts(bids_dict, skip_cols=bids_skip_cols)
bids_dicts_all.print()


BIDS events summary:
Summary for column dictionary :
  Categorical columns (2):
    cond_code (4 distinct values):
      0: 306
      1: 59410
      2: 55210
      3: 172519
    event_code (18 distinct values):
      1: 11703
      10: 4702
      11: 37548
      12: 37524
      13: 18778
      14: 18779
      199: 306
      2: 11701
      201: 29028
      202: 928
      255: 3
      3: 9296
      4: 9301
      5: 37171
      6: 37167
      7: 9406
      8: 9408
      9: 4696
  Value columns (0):


In [2]:
# from hed.tools.key_template import KeyTemplate
# print("Outputting the template:")
# key_columns = ['cond_code', 'event_code']
# template = KeyTemplate(key_columns)
# for file in bids_files:
#     template.update(file)
# template.resort()
# template.print()


In [3]:
print("Isolating the bad codes:")
for key, file in bids_dict.items():
    df_bids = get_new_dataframe(file)

    focus_mask = df_bids['cond_code'].map(str).isin(['1', '2'])
    shift_mask = df_bids['cond_code'].map(str).isin(['3'])
    focus_event_mask = df_bids['event_code'].map(str).isin(['3', '4', '5', '6'])
    shift_event_mask = df_bids['event_code'].map(str).isin(['7', '8', '9', '10', '11', '12', '13', '14'])
    bad_focus = sum(focus_mask & shift_event_mask)
    if bad_focus:
        status.add(key, f"{key} has {bad_focus} shift event codes in a focus condition")

    bad_shift = sum(shift_mask & focus_event_mask)
    if bad_shift:
        status.add(key, f"{key} has {bad_shift} focus event codes in a shift condition")

    bad_cond_mask = df_bids['cond_code'].map(str).isin(['0'])
    if sum(bad_cond_mask):
        status.add(key, f"{key} has {sum(bad_cond_mask)} cond_code values of 0")

    pulse_code_mask = df_bids['event_code'].map(str).isin(['199'])
    if sum(pulse_code_mask):
        status.add(key, f"{key} has {sum(pulse_code_mask)} event_code values of 199")

    pulse_combo_count = sum(pulse_code_mask & bad_cond_mask)
    if pulse_combo_count:
        status.add(key, f"{key} has {pulse_combo_count} event_code values of 199 with cond_code 0")

    unknown_count = sum(df_bids['event_code'].map(str).isin(['255']))
    if unknown_count:
        status.add(key, f"{key} has {unknown_count} event_code values of 255")

    pause_count = sum(df_bids['event_code'].map(str).isin(['202']))
    if pause_count:
        status.add(key, f"{key} has {pause_count} event_code values of 202")

Isolating the bad codes:


In [4]:
status.print_log()

sub-005_run-01
	sub-005_run-01 has 5 shift event codes in a focus condition
sub-007_run-01
	sub-007_run-01 has 3 event_code values of 255
sub-008_run-01
	sub-008_run-01 has 2874 shift event codes in a focus condition
sub-015_run-01
	sub-015_run-01 has 239 focus event codes in a shift condition
sub-020_run-01
	sub-020_run-01 has 109 cond_code values of 0
	sub-020_run-01 has 109 event_code values of 199
	sub-020_run-01 has 109 event_code values of 199 with cond_code 0
sub-021_run-01
	sub-021_run-01 has 130 cond_code values of 0
	sub-021_run-01 has 130 event_code values of 199
	sub-021_run-01 has 130 event_code values of 199 with cond_code 0
sub-022_run-01
	sub-022_run-01 has 67 cond_code values of 0
	sub-022_run-01 has 67 event_code values of 199
	sub-022_run-01 has 67 event_code values of 199 with cond_code 0
sub-036_run-02
	sub-036_run-02 has 721 focus event codes in a shift condition
